# ATP Upset Analysis

An **upset** = a match where the player with higher 52-week rolling points (`roll_points`) before the tournament lost.  
Pre-match ranking proxy: most recent `player_rankings.rank_date` ≤ `tournaments.start_date`.

In [1]:
import sqlite3
import pandas as pd

DB_PATH = '../atp/data/tennis.db'
conn = sqlite3.connect(DB_PATH)

In [2]:
# Load singles matches with tournament metadata and both player IDs
matches_sql = """
SELECT
    m.id          AS match_id,
    t.event_year,
    t.tournament_name,
    t.event_type,
    t.start_date,
    m.surface,
    m.round_short,
    m.winner_player_id,
    mp1.player_id AS p1_id,
    mp2.player_id AS p2_id,
    (p1.first_name || ' ' || p1.last_name) AS p1_name,
    (p2.first_name || ' ' || p2.last_name) AS p2_name
FROM matches m
JOIN tournaments t      ON m.tournament_id = t.id
JOIN match_players mp1  ON m.id = mp1.match_id AND mp1.is_player1 = 1
JOIN match_players mp2  ON m.id = mp2.match_id AND mp2.is_player1 = 0
JOIN players p1         ON mp1.player_id = p1.player_id
JOIN players p2         ON mp2.player_id = p2.player_id
WHERE m.is_doubles = 0
"""
matches = pd.read_sql(matches_sql, conn)
matches['start_date'] = pd.to_datetime(matches['start_date'])
print(f"Singles matches loaded: {len(matches):,}")
matches.head(3)

Singles matches loaded: 10,461


,match_id,event_year,tournament_name,event_type,start_date,surface,round_short,winner_player_id,p1_id,p2_id,p1_name,p2_name
0,1,2023,Adelaide International 1,250,2023-01-01,Hard,SF,d643,d643,mm58,Novak Djokovic,Daniil Medvedev
1,2,2023,Adelaide International 1,250,2023-01-01,Hard,QF,d643,d643,su55,Novak Djokovic,Denis Shapovalov
2,3,2023,Adelaide International 1,250,2023-01-01,Hard,QF,mm58,mm58,ke29,Daniil Medvedev,Karen Khachanov


In [3]:
# Load all player rankings (sorted by rank_date for merge_asof)
rankings = pd.read_sql(
    "SELECT player_id, rank_date, roll_points, roll_rank FROM player_rankings ORDER BY rank_date",
    conn
)
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])
print(f"Ranking rows loaded: {len(rankings):,}")

Ranking rows loaded: 213,433


In [4]:
def attach_pre_tourney_points(matches, rankings, player_col, suffix):
    """merge_asof to get most recent roll_points <= start_date for each player."""
    df = matches[['match_id', 'start_date', player_col]].copy()
    df = df.rename(columns={player_col: 'player_id'})
    df = df.sort_values('start_date')

    result = pd.merge_asof(
        df,
        rankings[['player_id', 'rank_date', 'roll_points', 'roll_rank']],
        left_on='start_date',
        right_on='rank_date',
        by='player_id',
        direction='backward'
    )
    return result[['match_id', 'roll_points', 'roll_rank']].rename(
        columns={'roll_points': f'roll_points_{suffix}', 'roll_rank': f'roll_rank_{suffix}'}
    )

p1_pts = attach_pre_tourney_points(matches, rankings, 'p1_id', 'p1')
p2_pts = attach_pre_tourney_points(matches, rankings, 'p2_id', 'p2')

df = matches.merge(p1_pts, on='match_id').merge(p2_pts, on='match_id')

# Drop rows where either player has no ranking data
before = len(df)
df = df.dropna(subset=['roll_points_p1', 'roll_points_p2'])
print(f"Matches with ranking data for both players: {len(df):,} (dropped {before - len(df):,})")

Matches with ranking data for both players: 10,461 (dropped 0)


In [5]:
# Identify favorite and flag upsets
# Favorite = player with higher roll_points before tournament
# Tie (equal points) → excluded from upset counting
df['favorite_id'] = df.apply(
    lambda r: r['p1_id'] if r['roll_points_p1'] > r['roll_points_p2']
              else (r['p2_id'] if r['roll_points_p2'] > r['roll_points_p1'] else None),
    axis=1
)
df['points_diff'] = abs(df['roll_points_p1'] - df['roll_points_p2'])

df_ranked = df[df['favorite_id'].notna()].copy()
df_ranked['is_upset'] = df_ranked['winner_player_id'] != df_ranked['favorite_id']

# Identify upset winner/loser names for display
df_ranked['fav_name'] = df_ranked.apply(
    lambda r: r['p1_name'] if r['favorite_id'] == r['p1_id'] else r['p2_name'], axis=1
)
df_ranked['und_name'] = df_ranked.apply(
    lambda r: r['p2_name'] if r['favorite_id'] == r['p1_id'] else r['p1_name'], axis=1
)
df_ranked['fav_points'] = df_ranked.apply(
    lambda r: r['roll_points_p1'] if r['favorite_id'] == r['p1_id'] else r['roll_points_p2'], axis=1
)
df_ranked['und_points'] = df_ranked.apply(
    lambda r: r['roll_points_p2'] if r['favorite_id'] == r['p1_id'] else r['roll_points_p1'], axis=1
)

total = len(df_ranked)
upsets = df_ranked['is_upset'].sum()
print(f"\n=== OVERALL ===")
print(f"Total matches (with clear favorite): {total:,}")
print(f"Upsets:                              {upsets:,}")
print(f"Upset rate:                          {upsets/total*100:.1f}%")


=== OVERALL ===
Total matches (with clear favorite): 10,455
Upsets:                              3,947
Upset rate:                          37.8%


In [6]:
# By year
by_year = df_ranked.groupby('event_year').agg(
    matches=('is_upset', 'count'),
    upsets=('is_upset', 'sum')
)
by_year['upset_rate'] = (by_year['upsets'] / by_year['matches'] * 100).round(1)
print("=== BY YEAR ===")
print(by_year.to_string())

=== BY YEAR ===
            matches  upsets  upset_rate
event_year                             
2023           2347     883        37.6
2024           2971    1101        37.1
2025           3562    1365        38.3
2026           1575     598        38.0


In [7]:
# By surface
by_surface = df_ranked.groupby('surface').agg(
    matches=('is_upset', 'count'),
    upsets=('is_upset', 'sum')
)
by_surface['upset_rate'] = (by_surface['upsets'] / by_surface['matches'] * 100).round(1)
print("=== BY SURFACE ===")
print(by_surface.sort_values('upset_rate', ascending=False).to_string())

=== BY SURFACE ===
         matches  upsets  upset_rate
surface                             
Clay        3308    1254        37.9
Hard        6011    2274        37.8
Grass       1105     406        36.7


In [8]:
# By tournament type
by_type = df_ranked.groupby('event_type').agg(
    matches=('is_upset', 'count'),
    upsets=('is_upset', 'sum')
)
by_type['upset_rate'] = (by_type['upsets'] / by_type['matches'] * 100).round(1)
print("=== BY TOURNAMENT TYPE ===")
print(by_type.sort_values('upset_rate', ascending=False).to_string())

=== BY TOURNAMENT TYPE ===
            matches  upsets  upset_rate
event_type                             
XXI              45      21        46.7
LVR              23      10        43.5
DCR              31      13        41.9
250            3163    1308        41.4
1000           2956    1107        37.4
GS             2346     829        35.3
500            1849     648        35.0
WC               42      11        26.2


In [9]:
# By round
by_round = df_ranked.groupby('round_short').agg(
    matches=('is_upset', 'count'),
    upsets=('is_upset', 'sum')
)
by_round['upset_rate'] = (by_round['upsets'] / by_round['matches'] * 100).round(1)
print("=== BY ROUND ===")
print(by_round.sort_values('upset_rate', ascending=False).to_string())

=== BY ROUND ===
             matches  upsets  upset_rate
round_short                             
NA                64      33        51.6
Q3               222     103        46.4
Q2               940     392        41.7
Q1              1467     595        40.6
R128            1169     453        38.8
Q-F              119      46        38.7
SF               392     149        38.0
RR                95      36        37.9
F                199      72        36.2
R32             2291     828        36.1
R16             1394     495        35.5
R64             1237     431        34.8
QF               732     253        34.6


In [10]:
# Upset rate by points-gap bucket
bins = [0, 500, 1000, 2000, 5000, float('inf')]
labels = ['0–500', '500–1000', '1000–2000', '2000–5000', '5000+']
df_ranked['gap_bucket'] = pd.cut(df_ranked['points_diff'], bins=bins, labels=labels)

by_gap = df_ranked.groupby('gap_bucket', observed=True).agg(
    matches=('is_upset', 'count'),
    upsets=('is_upset', 'sum')
)
by_gap['upset_rate'] = (by_gap['upsets'] / by_gap['matches'] * 100).round(1)
print("=== UPSET RATE BY POINTS GAP ===")
print(by_gap.to_string())

=== UPSET RATE BY POINTS GAP ===
            matches  upsets  upset_rate
gap_bucket                             
0–500          5556    2468        44.4
500–1000       1613     616        38.2
1000–2000      1408     453        32.2
2000–5000      1391     354        25.4
5000+           487      56        11.5


In [11]:
# Top 20 biggest upsets by points differential
top_upsets = (
    df_ranked[df_ranked['is_upset']]
    .nlargest(20, 'points_diff')
    [['event_year', 'tournament_name', 'round_short', 'surface',
      'fav_name', 'fav_points', 'und_name', 'und_points', 'points_diff']]
    .rename(columns={
        'fav_name': 'favorite', 'fav_points': 'fav_pts',
        'und_name': 'underdog', 'und_points': 'und_pts'
    })
    .reset_index(drop=True)
)
top_upsets.index += 1
print("=== TOP 20 BIGGEST UPSETS (by roll_points gap) ===")
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 120)
print(top_upsets.to_string())

=== TOP 20 BIGGEST UPSETS (by roll_points gap) ===
    event_year                              tournament_name round_short surface          favorite  fav_pts            underdog  und_pts  points_diff
1         2026                 Miami Open presented by Itau         R32    Hard    Carlos Alcaraz    13550     Sebastian Korda     1200        12350
2         2026                             BNP Paribas Open          SF    Hard    Carlos Alcaraz    13550     Daniil Medvedev     3360        10190
3         2025                          Rolex Paris Masters         R32    Hard    Carlos Alcaraz    11340      Cameron Norrie     1483         9857
4         2025                          Terra Wortmann Open         R16   Grass     Jannik Sinner    10880    Alexander Bublik     1225         9655
5         2025                       Rolex Shanghai Masters         R32    Hard     Jannik Sinner    10950   Tallon Griekspoor     1565         9385
6         2024                             BNP Paribas 

In [12]:
# Verification: Monte Carlo 2026 — Alcaraz (13590 pts) lost to Sinner (12400 pts)
mc2026 = df_ranked[
    (df_ranked['tournament_name'].str.contains('Monte', case=False)) &
    (df_ranked['event_year'] == 2026)
][['tournament_name', 'round_short', 'p1_name', 'roll_points_p1',
   'p2_name', 'roll_points_p2', 'fav_name', 'und_name', 'is_upset']]
print("=== VERIFICATION: Monte Carlo 2026 ===")
print(mc2026.to_string())

=== VERIFICATION: Monte Carlo 2026 ===
                 tournament_name round_short                  p1_name  roll_points_p1                  p2_name  roll_points_p2                 fav_name                 und_name  is_upset
10013  Rolex Monte-Carlo Masters           F           Carlos Alcaraz           13590            Jannik Sinner           12400           Carlos Alcaraz            Jannik Sinner      True
10015  Rolex Monte-Carlo Masters          QF        Valentin Vacherot            1831           Alex De Minaur            4095           Alex De Minaur        Valentin Vacherot      True
10016  Rolex Monte-Carlo Masters          SF         Alexander Zverev            5205            Jannik Sinner           12400            Jannik Sinner         Alexander Zverev     False
10017  Rolex Monte-Carlo Masters          QF             Joao Fonseca            1115         Alexander Zverev            5205         Alexander Zverev             Joao Fonseca     False
10019  Rolex Monte-Carlo M

In [13]:
conn.close()